# XGBoost-Algorithmus anwenden

Das ist die Musterlösung. XGBoost ist eine Gradient-Boosting-Bibliothek, die ein additives Ensemble aus flachen Bäumen aufbaut, wobei jeder neue Baum die Fehler der vorherigen korrigiert. Hier wenden wir sie auf den Diabetes-Datensatz der Pima an: Wir bereiten die Daten vor, trainieren einen `XGBClassifier`, evaluieren ihn, bilden dann die Entscheidungsfläche ab, die er gelernt hat, und visualisieren sie.

Der Datensatz wird ohne Spaltennamen geliefert, deshalb fügen wir sie nach dem Laden hinzu.

## Lernziele

Am Ende dieses Notebooks kannst du:

- einen XGBoost-Klassifikator auf einem echten medizinischen Datensatz trainieren.
- einen Klassifikator mit einem Classification Report und einer Confusion Matrix evaluieren.
- Feature Importances in eine Rangfolge bringen, um zu sehen, welche Eingaben die Vorhersagen bestimmen.
- die Entscheidungsfläche eines Modells über einem Raster von Feature-Werten abbilden und visualisieren.

## Import und Setup

In [ ]:
# Module importieren
from xgboost import XGBClassifier

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from itertools import product

RSEED=42

## Daten laden und vorbereiten

In [ ]:
# Daten importieren
df = pd.read_csv('../data/pima-native-americans-diabetes.csv', header=None)
column_names = ['pregnancies', 'glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi', 'diabetes_pedigree_function', 'age', 'outcome']
# Spaltennamen des DataFrames festlegen
df.columns = column_names

In [ ]:
df.head()

## Modell trainieren und evaluieren

XGBoost braucht wenig Vorbereitung: Es benötigt keine Feature-Skalierung und kommt mit den rohen Wertebereichen zurecht, wie sie sind. Wir halten 30% der Patientinnen zum Testen zurück, damit die Scores die Leistung auf Daten widerspiegeln, die das Modell nicht gesehen hat.

In [ ]:
# Daten aufteilen
X = df.drop('outcome', axis=1)
y = df['outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RSEED)

In [ ]:
# Modell auf den Trainingsdaten trainieren
xgb = XGBClassifier(random_state=RSEED,
                    n_jobs=-1,
                    #n_estimators=1000,
                    #learning_rate=0.3,
                    #subsample=0.5,
                    )
xgb.fit(X_train, y_train)
# Vorhersagen für den Testdatensatz erstellen
y_pred = xgb.predict(X_test)

In [ ]:
# Dein Modell evaluieren
print(classification_report(y_test, y_pred))

Das Modell erreicht bei den zurückgehaltenen Patientinnen eine Accuracy von etwa **0,71**. Aussagekräftiger ist bei diesem unbalancierten Datensatz die positive Klasse (Diabetes): eine Precision von etwa **0,57** und ein Recall von etwa **0,69**. Das Modell erkennt also ungefähr zwei Drittel der Patientinnen mit Diabetes, löst aber eine ganze Reihe von Fehlalarmen aus. Die Accuracy allein würde das verdecken, deshalb betrachten wir Precision und Recall pro Klasse.

In [ ]:
# Confusion Matrix anzeigen
cm = confusion_matrix(y_test, y_pred)

# Mit seaborn darstellen
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

Die Matrix zeigt etwa **110** True Negatives und **55** True Positives gegenüber **41** False Positives und **25** False Negatives. Die 25 False Negatives sind hier die teuren Fehler: Patientinnen mit Diabetes, die das Modell als gesund gekennzeichnet hat. Bei einem medizinischen Screening würdest du meist einige zusätzliche False Positives in Kauf nehmen, um den Recall zu erhöhen und diese Zahl zu senken.

## Entscheidungsfläche abbilden

Statt nur ein paar einzelne Werte vorherzusagen, ist es ein nützlicher Trick, Vorhersagen für eine ganze Matrix von Feature-Kombinationen zu erstellen, nicht nur für den Testdatensatz. Wenn wir das Modell über ein regelmäßiges Raster hinweg evaluieren, können wir seine **Entscheidungsfläche** ablesen: wie sich die vorhergesagte Wahrscheinlichkeit ändert, wenn sich die Eingaben ändern. Außerdem entsteht so eine schnelle Lookup-Tabelle, die das Modell annähert.

In [ ]:
#Funktion, die eine Zeile mit regelmäßigen Abständen erzeugt
def make_row(row_length,row_min,row_max,):
    step = (row_max - row_min)/row_length
    row = []
    for i in range(0,row_length+1):
#        print(df._STATE.min()+(step*i))
        row.append(row_min+(step*i))

    return row



In [ ]:
#Raster erzeugen. 
grid = pd.DataFrame()

for i in X_train.columns:
    row_min = X_train[i].min()
    row_max = X_train[i].max()
    row = make_row(3,row_min,row_max) #Ein hoher Wert hier führt im nächsten Schritt zu einer langen Wartezeit
    grid[i] = row

grid

In [ ]:
#Permutationen aller möglichen Werte erstellen

def expand_grid(dictionary):
   return pd.DataFrame([row for row in product(*dictionary.values())], 
                       columns=dictionary.keys())

dictionary = {}
for i in grid.columns:
    dictionary.update({i : grid[i]})

full_grid = expand_grid(dictionary)

full_grid


In [ ]:
#Vorhersagen für das vollständige Raster mit dem Modell erstellen
prob = xgb.predict_proba(full_grid)

#Positive Vorhersagen extrahieren und einer Spalte in full_grid zuweisen
full_grid['prob'] = prob[:,1]



In [ ]:
full_grid.tail(100)

Oben siehst du die letzten 100 Zeilen der Matrix, die wir erstellt haben, mit der berechneten Wahrscheinlichkeit für Diabetes. Diese Lookup-Tabelle ist eine Annäherung an das Modell, das wir erstellt haben. Sie ist zwar nicht so präzise wie das Modell, lässt sich aber für verschiedene Zwecke nutzen: als Plan B für dein Hauptmodell, wenn es offline geht, oder um die Darstellung der Welt zu visualisieren, die dein Modell aufgebaut hat.

In [ ]:
#Raster in einer CSV-Datei speichern: Vorsicht bei außergewöhnlich großen CSV-Dateien
full_grid.to_csv('../data/full_grid.csv')



## Feature Importance

XGBoost hält fest, wie viel jedes Feature über alle Bäume hinweg zur Verringerung der Unreinheit beigetragen hat. Eine Rangfolge dieser Beiträge zeigt, auf welche Messwerte sich das Modell am stärksten stützt.

In [ ]:
#Die wichtigsten Features aus dem Modell extrahieren
imp = list(zip(xgb.feature_names_in_,xgb.feature_importances_))
imp.sort(key = lambda tuple:tuple[1],reverse= True)
imp[0:2]

**Glucose** dominiert (Feature Importance etwa **0,28**), gefolgt von **age** und **BMI** (etwa **0,15** und **0,14**). Das passt zur klinischen Intuition: Die Vorhersagen des Modells hängen vor allem vom Blutzucker ab, mit Alter und Körpermasse als sekundären Signalen.

## Entscheidungsfläche in 3D visualisieren

Wir können nur in drei Dimensionen darstellen, deshalb konzentrieren wir uns auf die zwei wichtigsten Features und mitteln die vorhergesagte Wahrscheinlichkeit über die übrigen.

In [ ]:
#Wir können maximal 3 Variablen visualisieren, also gruppieren wir nach den 2 wichtigsten Variablen und berechnen den Mittelwert
grid_slice = full_grid.groupby(['glucose','bmi'],as_index=False).mean()

grid_slice

In [ ]:
#3D-Visualisierung als Plot der resultierenden Wahrscheinlichkeit vs die 2 wichtigsten Vorhersagevariablen erstellen
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

n = 100

xs = grid_slice.glucose
ys = grid_slice.bmi
zs = grid_slice.prob
ax.scatter3D(xs, ys, zs, marker='o', c = zs, cmap = 'rainbow')

ax.set_xlabel('glucose')
ax.set_ylabel('bmi')
ax.set_zlabel('probability')

plt.show()



Die Fläche steigt zu höherer Glukose und höherem BMI hin an: Das Modell weist in dieser Ecke des Feature-Raums eine höhere Wahrscheinlichkeit für Diabetes zu. Die Darstellung der Fläche ist eine Plausibilitätsprüfung: Sie zeigt, ob sich das Modell über den ganzen Eingabebereich sinnvoll verhält, nicht nur eine einzelne Accuracy-Zahl.

## Zusammenfassung

In diesem Notebook hast du:

- einen XGBoost-Klassifikator auf dem Diabetes-Datensatz der Pima trainiert.
- ihn mit einem Classification Report und einer Confusion Matrix evaluiert und dabei Precision und Recall pro Klasse bei unbalancierten Daten abgelesen.
- die Feature Importances in eine Rangfolge gebracht und Glukose, Alter und BMI als die stärksten Prädiktoren erkannt.
- die Entscheidungsfläche über einem Feature-Raster abgebildet und in 3D visualisiert.

## Quellen & weiterführende Literatur

- [**XGBoost Documentation**](https://xgboost.readthedocs.io/en/stable/): Offizielle Dokumentation der Gradient-Boosting-Bibliothek.
- [**How to Develop Your First XGBoost Model in Python**](https://machinelearningmastery.com/develop-first-xgboost-model-python-scikit-learn/): Eine Schritt-für-Schritt-Anleitung zum Training von XGBoost mit scikit-learn.
- [**Scikit-learn: Metrics and Scoring**](https://scikit-learn.org/stable/modules/model_evaluation.html): Wie Accuracy, Precision, Recall und F-Score definiert sind und wann du welche Metrik verwendest.